In [53]:
import pandas as pd
import numpy as np
import warnings
import msoffcrypto
import io
import re
warnings.filterwarnings("ignore")

pd.set_option('display.max_colwidth', None)  # Display full content of each column
pd.set_option('display.max_columns', None)   # Display all columns
pd.set_option('display.width', 5000)         # Set display width

In [ ]:
import os

password = os.environ.get("BANK_STATEMENT_PASSWORD")  # set this in your local shell/.env, never hardcode it here

#Add Multiple csvs or just one csv to the list below. The script will merge all of them into one single csv.
files = [ "CSVS\\SBI\\2024.xlsx", "CSVS\\SBI\\2025.xlsx","CSVS\\SBI\\2026.xlsx", "CSVS\\SBI\\2027.xlsx"]
# files = ["CSVS\\SBI\\2027_pswd.xlsx"]


Protected CSV with Addition Info -> Clean CSV 
--
 

In [55]:

# def truncate_at_first_blank_row(df):
#     """Each yearly export has its own trailing legend/footer section after the
#     real transactions, starting at the first fully-blank row. Must be applied
#     per file, before concatenation — applying it to the merged dataframe would
#     cut off every file after the first one's footer."""
#     blank_rows = df.apply(
#         lambda row: all(pd.isna(x) or str(x).strip() == "" for x in row),
#         axis=1
#     )
#     if blank_rows.any():
#         first_blank = blank_rows[blank_rows].index[0]
#         df = df.iloc[:first_blank]
#     return df.reset_index(drop=True)


# dfs = []

# for file_name in files:
#     decrypted = io.BytesIO()

#     with open(file_name, "rb") as f:
#         office_file = msoffcrypto.OfficeFile(f)
#         office_file.load_key(password=password)
#         office_file.decrypt(decrypted)

#     df = pd.read_excel(decrypted)
#     df = truncate_at_first_blank_row(df)
#     dfs.append(df)

# final_df = pd.concat(dfs, ignore_index=True)
# final_df.to_excel("CSVS\\SBI\\2027.xlsx", index=False)

In [56]:
def truncate_at_first_blank_row(df):
    """Each yearly export has its own trailing legend/footer section after the
    real transactions, starting at the first fully-blank row. Must be applied
    per file, before concatenation — applying it to the merged dataframe would
    cut off every file after the first one's footer."""
    blank_rows = df.apply(
        lambda row: all(pd.isna(x) or str(x).strip() == "" for x in row),
        axis=1
    )
    if blank_rows.any():
        first_blank = blank_rows[blank_rows].index[0]
        df = df.iloc[:first_blank]
    return df.reset_index(drop=True)

def is_encrypted(file_path: str) -> bool:
    with open(file_path, "rb") as f:
        office_file = msoffcrypto.OfficeFile(f)
        return office_file.is_encrypted()


DATE_FORMAT_CANDIDATES = [
    "%d/%m/%Y", "%d-%m-%Y", "%d/%m/%y", "%d-%m-%y",
    "%m/%d/%Y", "%m-%d-%Y",
    "%Y-%m-%d", "%Y/%m/%d",
]

def detect_date_format(date_series: pd.Series, candidates=DATE_FORMAT_CANDIDATES) -> str:
    """Bank exports mix DD/MM/YYYY, MM/DD/YYYY, YYYY-MM-DD across banks/years.
    Ambiguous dates (day <= 12) parse 'successfully' under multiple formats,
    so check the WHOLE column, not a small sample — a day > 12 somewhere in
    a full year's statement is what actually disambiguates DD/MM vs MM/DD."""
    values = date_series.dropna().astype(str).str.strip()
    values = values[values != ""]
    if values.empty:
        raise ValueError("No non-null dates to detect format from")

    for fmt in candidates:
        parsed = pd.to_datetime(values, format=fmt, errors="coerce")
        if parsed.notna().all():
            return fmt

    raise ValueError(f"Could not detect a consistent date format. Sample values: {values.head(10).tolist()}")


def parse_dates(date_series: pd.Series) -> pd.Series:
    fmt = detect_date_format(date_series)
    print(f"  detected date format: {fmt}")
    return pd.to_datetime(date_series.astype(str).str.strip(), format=fmt, errors="coerce")




def _split_label_value(cell_text: str) -> dict:
    """Parse 'Label  :  Value' lines out of a header cell into {label: value}."""
    result = {}
    for line in str(cell_text).split("\n"):
        line = line.strip()
        if ":" not in line:
            continue
        label, _, value = line.partition(":")
        label = re.sub(r"\s+", " ", label.strip().lower())
        value = value.strip()
        if label and value:
            result[label] = value
    return result


def find_table_start_row(df_no_header: pd.DataFrame, max_scan: int = 40) -> int:
    """Locate the real 'Date | Details | ... | Balance' row instead of assuming
    a fixed row count — header length varies by bank/export/year."""
    for i in range(min(max_scan, len(df_no_header))):
        row_values = [str(x).strip().lower() for x in df_no_header.iloc[i].tolist()]
        if "date" in row_values and "balance" in row_values:
            return i
    raise ValueError("Could not locate transaction table header row (Date/.../Balance)")


def mask_identifier(value: str, keep_last: int = 4) -> str:
    """Mask a sensitive numeric identifier, keeping only the last few digits
    so the source account is still distinguishable without exposing it."""
    digits = re.sub(r"\D", "", str(value))
    if not digits:
        return None
    if len(digits) <= keep_last:
        return "X" * len(digits)
    return "X" * (len(digits) - keep_last) + digits[-keep_last:]

def parse_statement_header(decrypted_stream: io.BytesIO) -> dict:
    """Extract account-holder metadata from the header block above the
    transaction table. '_raw_sensitive' is for debugging only in this
    notebook session — never write it to a CSV/DataFrame that gets saved."""
    decrypted_stream.seek(0)
    raw = pd.read_excel(decrypted_stream, header=None)

    table_start = find_table_start_row(raw)
    header_block = raw.iloc[:table_start]

    fields = {}
    email_match = None
    name_match = None
    for _, row in header_block.iterrows():
        row_text = " ".join(str(x) for x in row.tolist() if pd.notna(x))
        if email_match is None:
            candidate = re.search(r"[\w.\-]+@[\w.\-]+", row_text)
            if candidate:
                email_match = candidate
                name_match = re.match(r"\s*(?:Mr\.|Mrs\.|Ms\.)?\s*([A-Za-z .]+)", row_text)
        for cell in row.tolist():
            if pd.notna(cell):
                fields.update(_split_label_value(cell))

    account_number = fields.get("account number")

    metadata = {
        "table_start_row": table_start,
        "account_holder_name": name_match.group(1).strip() if name_match else None,
        "masked_account_number": mask_identifier(account_number),
        "nominee_name": fields.get("nominee name"),
        "statement_period": fields.get("statement from"),
        "_raw_sensitive": {
            "email": email_match.group(0) if email_match else None,
            "account_number": account_number,
            "cif_number": fields.get("cif number"),
            "micr_code": fields.get("micr code"),
            "ifsc_code": fields.get("ifsc code"),
            "branch_phone": fields.get("branch phone"),
            "branch_email_id": fields.get("branch email id"),
        },
    }
    return metadata


In [57]:
dfs = []
account_metadata = []  # safe fields only, one entry per file

for file_name in files:
    decrypted = io.BytesIO()

    with open(file_name, "rb") as f:
        office_file = msoffcrypto.OfficeFile(f)
        if office_file.is_encrypted():
            office_file.load_key(password=password)
            office_file.decrypt(decrypted)
        else:
            f.seek(0)
            decrypted.write(f.read())

    meta = parse_statement_header(decrypted)
    print(file_name, "->", meta["account_holder_name"], meta["masked_account_number"])

    decrypted.seek(0)
    df = pd.read_excel(decrypted, skiprows=meta["table_start_row"])
    df = truncate_at_first_blank_row(df)

    df["Date"] = parse_dates(df["Date"])
    # df["account_holder_name"] = meta["account_holder_name"]
    # df["masked_account_number"] = meta["masked_account_number"]

    account_metadata.append({k: v for k, v in meta.items() if k != "_raw_sensitive"})
    dfs.append(df)

final_df = pd.concat(dfs, ignore_index=True)

print(final_df.head(3))
print(account_metadata)
final_df["Details"] = final_df["Details"].astype(str).str.strip()
final_df["Balance"] = pd.to_numeric(final_df["Balance"].astype(str).str.replace(r"[^\d\.\-]", "", regex=True), errors="coerce")

before = len(final_df)
final_df = final_df.drop_duplicates(subset=["Date", "Details", "Balance"], keep="first").reset_index(drop=True)
dropped = before - len(final_df)
print(f"Dropped {dropped} duplicate rows from overlapping yearly exports ({before} -> {len(final_df)})")

final_df.to_excel("CSVS\\SBI\\SpendWise_4yrs_RAW.xlsx", index=False)


CSVS\SBI\2024.xlsx -> Yash Sameer Sawant XXXXXXX7686
  detected date format: %d/%m/%Y
CSVS\SBI\2025.xlsx -> Yash Sameer Sawant XXXXXXX7686
  detected date format: %d/%m/%Y
CSVS\SBI\2026.xlsx -> Yash Sameer Sawant XXXXXXX7686
  detected date format: %d/%m/%Y
CSVS\SBI\2027.xlsx -> Yash Sameer Sawant XXXXXXX7686
  detected date format: %d/%m/%Y
        Date                                                                                                        Details  Ref No/Cheque No   Debit  Credit  Balance
0 2023-04-01         DEP TFR   INB IMPS309111290781/9890160567/XX8237/\n Son   0098030162099 AT 71097 EVERSHINE CITY BRANCH               NaN     NaN  1500.0  1614.48
1 2023-04-01   WDL TFR   UPI/DR/309122218462/ASIM HEM/HDFC/asimshah\n 21/UPI   0099744162096 AT 71097 EVERSHINE CITY BRANCH               NaN    75.0     NaN  1539.48
2 2023-04-02   WDL TFR   UPI/DR/309252494278/SAI RAJ /PYTM/paytmqr2\n 81/UPI   0096407162097 AT 71097 EVERSHINE CITY BRANCH               NaN  1102.0     N

In [63]:
final_df

,Date,Details,Ref No/Cheque No,Debit,Credit,Balance,is_self_transfer
0,2023-04-01,DEP TFR INB IMPS309111290781/9890160567/XX8237/\n Son 0098030162099 AT 71097 EVERSHINE CITY BRANCH,NaN,NaN,1500.0,1614.48,False
1,2023-04-01,WDL TFR UPI/DR/309122218462/ASIM HEM/HDFC/asimshah\n 21/UPI 0099744162096 AT 71097 EVERSHINE CITY BRANCH,NaN,75.0,NaN,1539.48,False
2,2023-04-02,WDL TFR UPI/DR/309252494278/SAI RAJ /PYTM/paytmqr2\n 81/UPI 0096407162097 AT 71097 EVERSHINE CITY BRANCH,NaN,1102.0,NaN,437.48,False
3,2023-04-02,WDL TFR UPI/DR/309255302589/PRACHI S/SBIN/prachi24\n 77/UPI 0099846162090 AT 71097 EVERSHINE CITY BRANCH,NaN,10.0,NaN,427.48,False
4,2023-04-02,DEP TFR UPI/CR/309220310152/PRACHI S/SBIN/prachisw\n t2/Pay 0098828162099 AT 71097 EVERSHINE CITY BRANCH,NaN,NaN,1500.0,1927.48,False
...,...,...,...,...,...,...,...
1913,2026-06-04,WDL TFR UPI/DR/615578484258/Compass /YESB/paytm-333\n 4/Sent 0097693162093 AT 71097 EVERSHINE CITY BRANCH,NaN,33.0,NaN,3875.16,False
1914,2026-06-04,WDL TFR UPI/DR/615527273989/Sonu/YESB/paytm.s1cz\n /UPI 0097693162093 AT 71097 EVERSHINE CITY BRANCH,NaN,90.0,NaN,3785.16,False
1915,2026-06-04,WDL TFR UPI/DR/307418476563/SEJAL SI/ICIC/singhseja\n l/Food 0097693162093 AT 71097 EVERSHINE CITY BRANCH,NaN,170.0,NaN,3615.16,False
1916,2026-06-06,WDL TFR UPI/DR/207846219220/Daily Fr/YESB/paytmqr1q\n 0/Dinn 0097695162091 AT 71097 EVERSHINE CITY BRANCH,NaN,75.0,NaN,3540.16,False


In [59]:
print(account_metadata)

[{'table_start_row': 17, 'account_holder_name': 'Yash Sameer Sawant', 'masked_account_number': 'XXXXXXX7686', 'nominee_name': 'Sameer sawant', 'statement_period': '01-04-2023  to  31-03-2024'}, {'table_start_row': 17, 'account_holder_name': 'Yash Sameer Sawant', 'masked_account_number': 'XXXXXXX7686', 'nominee_name': 'Sameer sawant', 'statement_period': '01-04-2024  to  31-03-2025'}, {'table_start_row': 17, 'account_holder_name': 'Yash Sameer Sawant', 'masked_account_number': 'XXXXXXX7686', 'nominee_name': 'Sameer sawant', 'statement_period': '01-04-2025  to  14-03-2026'}, {'table_start_row': 17, 'account_holder_name': 'Yash Sameer Sawant', 'masked_account_number': 'XXXXXXX7686', 'nominee_name': 'Sameer sawant', 'statement_period': '01-01-2026  to  08-06-2026'}]


In [62]:
def is_self_transfer(details: str, account_holder_name: str, nominee_name: str = None) -> bool:
    if not isinstance(details, str) or not account_holder_name:
        return False
    names = [n for n in [account_holder_name, nominee_name] if n]
    tokens = set()
    for n in names:
        tokens.update(t.lower() for t in re.split(r"\s+", n) if len(t) > 2)
    details_lower = details.lower()
    return any(t in details_lower for t in tokens)

final_df["is_self_transfer"] = final_df.apply(
    lambda r: is_self_transfer(r["Details"], account_metadata[0]["account_holder_name"]), axis=1
)
final_df[final_df["is_self_transfer"]=='True']

,Date,Details,Ref No/Cheque No,Debit,Credit,Balance,is_self_transfer


In [ ]:
# final_df.to_csv("CSVS\\SBI\\SpendWise_4yrs_RAW.csv", index=False)

In [66]:
test=pd.read_excel("CSVS\\SBI\\SpendWise_4yrs_RAW.xlsx")
test

,Date,Details,Ref No/Cheque No,Debit,Credit,Balance
0,2023-04-01,DEP TFR INB IMPS309111290781/9890160567/XX8237/\n Son 0098030162099 AT 71097 EVERSHINE CITY BRANCH,NaN,NaN,1500.0,1614.48
1,2023-04-01,WDL TFR UPI/DR/309122218462/ASIM HEM/HDFC/asimshah\n 21/UPI 0099744162096 AT 71097 EVERSHINE CITY BRANCH,NaN,75.0,NaN,1539.48
2,2023-04-02,WDL TFR UPI/DR/309252494278/SAI RAJ /PYTM/paytmqr2\n 81/UPI 0096407162097 AT 71097 EVERSHINE CITY BRANCH,NaN,1102.0,NaN,437.48
3,2023-04-02,WDL TFR UPI/DR/309255302589/PRACHI S/SBIN/prachi24\n 77/UPI 0099846162090 AT 71097 EVERSHINE CITY BRANCH,NaN,10.0,NaN,427.48
4,2023-04-02,DEP TFR UPI/CR/309220310152/PRACHI S/SBIN/prachisw\n t2/Pay 0098828162099 AT 71097 EVERSHINE CITY BRANCH,NaN,NaN,1500.0,1927.48
...,...,...,...,...,...,...
1913,2026-06-04,WDL TFR UPI/DR/615578484258/Compass /YESB/paytm-333\n 4/Sent 0097693162093 AT 71097 EVERSHINE CITY BRANCH,NaN,33.0,NaN,3875.16
1914,2026-06-04,WDL TFR UPI/DR/615527273989/Sonu/YESB/paytm.s1cz\n /UPI 0097693162093 AT 71097 EVERSHINE CITY BRANCH,NaN,90.0,NaN,3785.16
1915,2026-06-04,WDL TFR UPI/DR/307418476563/SEJAL SI/ICIC/singhseja\n l/Food 0097693162093 AT 71097 EVERSHINE CITY BRANCH,NaN,170.0,NaN,3615.16
1916,2026-06-06,WDL TFR UPI/DR/207846219220/Daily Fr/YESB/paytmqr1q\n 0/Dinn 0097695162091 AT 71097 EVERSHINE CITY BRANCH,NaN,75.0,NaN,3540.16
